# Retail Customer Purchase Prediction

**Abstract** *(fill after results)*

**Acknowledgments** *(course, instructor, collaborators)*

## Background
Why purchase intent matters; prior work; interpretability vs performance.

## Data
- Primary: UCI Online Shoppers (≈12k; label `Revenue`)
- Optional: Retail II (RFM aggregation)
- Leakage guardrails: features known at/before session end.

In [ ]:
import os, pandas as pd, numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

DATA_DIR = Path('../data')
OUT_FIG = Path('../outputs/figures'); OUT_FIG.mkdir(parents=True, exist_ok=True)
OUT_TAB = Path('../outputs/tables'); OUT_TAB.mkdir(parents=True, exist_ok=True)
plt.rcParams['figure.dpi'] = 150
print('Data dir:', DATA_DIR)

In [ ]:
uci = pd.read_csv(DATA_DIR/'online_shoppers_intention.csv')
print(uci.shape); uci.head()

## EDA
Class balance; key distributions.

In [ ]:
target='Revenue'
counts = uci[target].value_counts().sort_index()
display(counts); (counts/counts.sum()).rename('proportion')

In [ ]:
ax = counts.plot(kind='bar', title='Class Counts'); ax.get_figure().tight_layout()
ax.get_figure().savefig(OUT_FIG/'class_counts.png')

## Preprocessing & Split

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

X = uci.drop(columns=[target]); y = uci[target].astype(int)
num = X.select_dtypes(include=['int64','float64']).columns.tolist()
cat = [c for c in X.columns if c not in num]

pre = ColumnTransformer([('num', StandardScaler(), num), ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X, y, test_size=0.30, stratify=y, random_state=42)
X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42)
len(X_tr), len(X_va), len(X_te)

## Models & Metrics

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score, RocCurveDisplay, confusion_matrix

def fit_eval(clf, name):
    pipe = Pipeline([('pre', pre), ('clf', clf)]); pipe.fit(X_tr, y_tr)
    try: proba = pipe.predict_proba(X_va)[:,1]
    except Exception: proba = pipe.decision_function(X_va)
    pred = (proba>=0.5).astype(int)
    return pipe, {'model': name, 'roc_auc_val': roc_auc_score(y_va, proba),
                  'f1_val': f1_score(y_va, pred),
                  'precision_val': precision_score(y_va, pred, zero_division=0),
                  'recall_val': recall_score(y_va, pred)}
    
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'DecisionTree': DecisionTreeClassifier(),
    'RandomForest': RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=42)
}
fitted, results = {}, []
for n, m in models.items():
    p, met = fit_eval(m, n); fitted[n]=p; results.append(met)

res = pd.DataFrame(results).sort_values('roc_auc_val', ascending=False); display(res)
res.to_csv(OUT_TAB/'validation_metrics.csv', index=False)

## Test Evaluation (Lock Best)

In [ ]:
best = res.iloc[0]['model']; pipe = fitted[best]
try: proba = pipe.predict_proba(X_te)[:,1]
except Exception: proba = pipe.decision_function(X_te)
pred = (proba>=0.5).astype(int)
test = {'model': best, 'roc_auc_test': roc_auc_score(y_te, proba),
        'f1_test': f1_score(y_te, pred),
        'precision_test': precision_score(y_te, pred, zero_division=0),
        'recall_test': recall_score(y_te, pred)}
pd.DataFrame([test]).to_csv(OUT_TAB/'test_metrics.csv', index=False); test

## ROC & Confusion Matrix

In [ ]:
RocCurveDisplay.from_predictions(y_te, proba)
import matplotlib.pyplot as plt
plt.title(f'ROC Curve — Test ({best})'); plt.tight_layout(); plt.savefig(OUT_FIG/'roc_curve_test.png')

In [ ]:
import seaborn as sns
cm = confusion_matrix(y_te, pred)
fig, ax = plt.subplots(); sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual'); ax.set_title(f'Confusion Matrix — Test ({best})')
fig.tight_layout(); fig.savefig(OUT_FIG/'confusion_matrix_test.png')

## Ablations (placeholders)
Compare feature groups and model families; export CSVs/plots.

In [ ]:
# TODO: implement feature-group masks and re-train; save to OUT_TAB/'ablations.csv'

## Business Takeaways
Signals that move purchase likelihood; precision/recall trade-offs.

## Conclusion & Future Work
Calibration, thresholds, cost-sensitive metrics, deployment hooks.

## Reproducibility Notes
Random seeds, versions, environment; data provenance.